## Quantization & Distillation

In this lesson, we will talk a bit about distillation, take a detailed look at the quantization process, implementing it from scratch. Then we will add quantization to DistilBERT and measure improvements regarding inference speed and memory usage.

### Plan

0. Distillation
1. Dataset and model
2. Implementation of quantization
3. Quantization in torch
4. Quantization in huggingface
5. PEFT for quantized models

In [ ]:
# Environment for week12: Quantization (int8 from scratch, torch, bitsandbytes, ONNX static), GPTQ, QLoRA.
# Colab or a fresh virtualenv: run this cell once. Locally: `uv sync --group week12` in the repo root instead.
# Optional (Linux + NVIDIA GPU): %pip install -q "bitsandbytes>=0.45" for the 8-bit/4-bit sections.
%pip install -q "torch>=2.4,<3" "transformers>=4.56,<5" "datasets>=3.6,<4" "evaluate>=0.4.3" "accelerate>=1.0" "peft>=0.13,<1" "tqdm>=4.66" "optimum-onnx[onnxruntime]>=0.1" "onnxruntime>=1.19"

## 0. Distillation

![Alt text](image.png)

**Why we use distillation**

* reduce model size without major quality loss
* speed up inference
* deploy models on edge devices
* transfer capabilities of large LLMs into smaller ones


**How it works**

*1) The teacher produces guidance*

The large model outputs:

* **logits** (raw scores before softmax)
* **probability distributions** over classes or tokens
* sometimes **intermediate representations** (hidden states, attention)


*2) The student learns to imitate*

The student is trained with two loss terms:

a) Imitation loss (KL divergence)

The student’s softmax should match the teacher’s softmax:

$$L_{KD} = T^2 KL(\text{softmax}(z_t/T) |\text{softmax}(z_s/T))$$

* $z_t, z_s$: teacher and student logits
* $T$: temperature — softens the distribution

Soft distributions reveal **relationships between classes**, not just the winner.

b) Cross-entropy loss with true labels

$$L = \alpha L_{CE} + (1-\alpha) L_{KD}$$

The teacher doesn’t just give the final answer — it reveals how confident it is and which alternatives are close.
The student absorbs this nuanced structure, which results in better generalization.


*In LLM context*

There are two options: we either try to replicate the token logits distribution:

* Teacher: GPT-4/5 or a large proprietary model
* Student: 1B–7B parameter model
* Goal: mimic teacher token distributions

Or teacher can also generate synthetic training data and via training on it wereplicate its behaviour (models like Zephyr, Orca, etc). Trying to replicate the reasoning is also considered a distillation.

## 1. Data

We will work on the task of sentiment analysis on the IMDb dataset.

In [6]:
from datasets import load_dataset

imdb = load_dataset("imdb", split='test')

/home/amshabalin/anaconda3/envs/karpov/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import AutoTokenizer

# a DistilBERT fine-tuned on IMDb, from the Hub
CHECKPOINT = "lvwerra/distilbert-imdb"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

In [8]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_imdb = imdb.map(preprocess_function, batched=True, remove_columns=['text'])

In [9]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [10]:
from torch.utils.data import DataLoader

dataloader = DataLoader(tokenized_imdb, collate_fn=data_collator, batch_size=64, shuffle=False)

In [11]:
batch = next(iter(dataloader))

## Model

We will use the distilled version of BERT, `distilbert-base-uncased`, as our training model. We take a copy fine-tuned on IMDb from the Hub (`lvwerra/distilbert-imdb`).

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2).to(device)

In [13]:
total_number_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_number_params

66955010

In [14]:
print('Memory usage: %.3f GB' % (model.get_memory_footprint() / 2**30))

Модель занимает 0.249 GB


We measure quality so that we have something to compare with after quantization.

**!NB** In your theses you also should do evaluation of baseline and your method, to show the performance improvement.

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(model, device=device):
    model.eval()

    correct = 0
    for batch in tqdm(dataloader):
        output = model(batch['input_ids'].to(device), batch['attention_mask'].to(device)).logits.cpu()
        correct += (batch['labels'] == output.argmax(-1)).float().sum()

    accuracy = correct / len(dataloader.dataset)
    return accuracy

In [13]:
evaluate(model)

100%|██████████| 391/391 [02:04<00:00,  3.14it/s]


tensor(0.9313)

## Quantization

Let's recall that quantization from `fp` to `int` is performed using the following process:

$$
x = s(x_q - z) \qquad x_q = \text{round}\bigg(\frac{x}{s} + z\bigg)
$$

In this case, we can derive quantized matrix multiplication for a linear layer. Note that only quantized numbers are under the sum sign, for which all operations are faster.


\begin{aligned}
& Y_{i, j} \\
& = b_j+\sum_{k=1}^d X_{i, k} W_{k, j} \\
& = s_b\left(b_{q, j}-z_b\right)+\sum_{k=1}^d s_X\left(X_{q, i, k}-z_X\right) s_W\left(W_{q, k, j}-z_W\right) \\
& = s_b\left(b_{q, j}-z_b\right)+s_X s_W \sum_{k=1}^d\left(X_{q, i, k}-z_X\right)\left(W_{q, k, j}-z_W\right) \\
& = s_b\left(b_{q, j}-z_b\right)+s_X s_W\left[\left(\sum_{k=1}^d X_{q, i, k} W_{q, k, j}\right)-\left(z_W \sum_{k=1}^d X_{q, i, k}\right)-\left(z_X \sum_{k=1}^d W_{q, k, j}\right)+ d z_X z_W\right] \\
& = s_Y\left(Y_{q, i, j}-z_Y\right)
\end{aligned}

All that remains is to express $Y_{q, i, j}$ from here.
\begin{aligned}
Y_{q, i, j} = round\Bigg( z_Y & +\frac{s_b}{s_Y}\left(b_{q, j}-z_b\right) \\
& +\frac{s_X s_W}{s_Y}\left[\left(\sum_{k=1}^d X_{q, i, k} W_{q, k, j}\right)-\left(z_W \sum_{k=1}^d X_{q, i, k}\right)-\left(z_X \sum_{k=1}^d W_{q, k, j}\right)+d z_X z_W\right] \Bigg)
\end{aligned}

Since all those operations betweet `[` and `]` are in int8 and are considered to be fast, we'll calculate everything a bit quicker.

Now let's implement this quantization in `torch`.

In [16]:
from torch import nn

class QuantizedTensor(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()
        # these are min-max for int8 quantization
        
        self.a_q = -128
        self.b_q = 127

        # добавляем микро сдвиг, чтобы не делить на 0 в случае чего
        a, b = x.min() - 1e-6, x.max() + 1e-6

        self.s, self.z = self.quantization_constants(a, b)
        self.x_q = nn.Parameter(self.quantize(x).detach(), requires_grad=False)

    def quantize(self, x):
        x_q = torch.round(x / self.s + self.z)
        
        # clip if the value is out of range
        x_q = torch.clip(x_q, min=self.a_q, max=self.b_q)

        return x_q.to(torch.int8)

    def dequantize(self):
        x_q = self.x_q.to(torch.int32)
        x = self.s * (x_q - self.z)

        return x
    
    # calculatinz s & z from the formulas above
    def quantization_constants(self, a, b):
        s = (b - a) / (self.b_q - self.a_q)
        z = int((b * self.a_q - a * self.b_q) / (b - a))

        return s, z

Now we can work with the matrices, since they also are tensors.

In [17]:
class QuantizedLinear(nn.Module):
    def __init__(self, module: nn.Linear):
        super().__init__()
        self.weight = QuantizedTensor(module.weight.data.T)
        if module.bias is not None:
            self.bias = QuantizedTensor(module.bias.data)
        else:
            self.bias = None

    def forward(self, x: torch.Tensor):
        d = self.weight.x_q.shape[-1]

        x = QuantizedTensor(x)

        if self.bias is not None:
            bias = self.bias.s * (self.bias.x_q - self.bias.z)
        else:
            bias = 0

        # переводим все в int32, чтобы не было переполнения
        product = (x.s * self.weight.s) * (
            torch.matmul(x.x_q.to(torch.int32), self.weight.x_q.to(torch.int32)) - \
            self.weight.z * torch.sum(x.x_q.to(torch.int32), axis=-1, keepdims=True) - \
            x.z * torch.sum(self.weight.x_q.to(torch.int32), axis=-2, keepdims=True) + \
            d * x.z * self.weight.z
        )
        return product + bias

### Testing

In [18]:
lin = nn.Linear(128, 128)
q_lin = QuantizedLinear(lin)

In [19]:
x = torch.rand(256, 128)

In [20]:
%%timeit

with torch.no_grad():
    lin(x)

51 μs ± 9.41 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [21]:
%%timeit

with torch.no_grad():
    q_lin(x)

1.23 ms ± 123 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


why it's slower than original?

`torch` is not optimized for matrix multiplication with the `int` type, so the quantized layer runs slower.

In [22]:
expected_output = lin(x)
output = q_lin(x)

torch.abs(expected_output - output).mean().item()

0.0011311459820717573

The error is neglectible, hence we did everything right.

In [27]:
def get_layer(model, name):
    layer = model
    for attr in name.split("."):
        layer = getattr(layer, attr)
    return layer

def set_layer(model, name, layer):
    if len(name.rsplit(".", 1)) > 1:
        attrs, name = name.rsplit(".", 1)
        model = get_layer(model, attrs)

    setattr(model, name, layer)

for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        module = QuantizedLinear(module)
        
        set_layer(model, name, module)

In [28]:
print('Memory usage: %.3f GB' % (model.get_memory_footprint() / 2**30))

Модель занимает 0.129 GB


With this quantization, we've reduced memory costs by half. We could have achieved even more if we had quantized embeddings as well, for example.

## Torch quantization

To avoid implementing quantization yourself, you can use the ready-made version from `torch`. It implements both dynamic and static quantization, as well as various calibration methods. However, it also has a number of drawbacks, which we will discuss later.

In [130]:
import torch

In [ ]:
# a fresh float32 copy on the CPU: torch dynamic quantization runs on the CPU only
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2)

In [132]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [27]:
%%time

with torch.no_grad():
    output = model(batch['input_ids'], batch['attention_mask'])

CPU times: user 38.3 s, sys: 6.77 s, total: 45.1 s
Wall time: 24.5 s


In [ ]:
model_int8 = torch.ao.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8  # a special type that also contains quantization coefficients
)
print('Memory usage: %.3f GB' % (model_int8.get_memory_footprint() / 2**30))

In fact, there are more, but quantized weights are not displayed in the list of model parameters.

In [29]:
model_int8

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (k_lin): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (v_lin): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (out_lin): DynamicQuantizedLinear(in_features=768, out_feature

In [30]:
for name, p in model_int8.named_parameters():
    print(name)

distilbert.embeddings.word_embeddings.weight
distilbert.embeddings.position_embeddings.weight
distilbert.embeddings.LayerNorm.weight
distilbert.embeddings.LayerNorm.bias
distilbert.transformer.layer.0.sa_layer_norm.weight
distilbert.transformer.layer.0.sa_layer_norm.bias
distilbert.transformer.layer.0.output_layer_norm.weight
distilbert.transformer.layer.0.output_layer_norm.bias
distilbert.transformer.layer.1.sa_layer_norm.weight
distilbert.transformer.layer.1.sa_layer_norm.bias
distilbert.transformer.layer.1.output_layer_norm.weight
distilbert.transformer.layer.1.output_layer_norm.bias
distilbert.transformer.layer.2.sa_layer_norm.weight
distilbert.transformer.layer.2.sa_layer_norm.bias
distilbert.transformer.layer.2.output_layer_norm.weight
distilbert.transformer.layer.2.output_layer_norm.bias
distilbert.transformer.layer.3.sa_layer_norm.weight
distilbert.transformer.layer.3.sa_layer_norm.bias
distilbert.transformer.layer.3.output_layer_norm.weight
distilbert.transformer.layer.3.outpu

Now we measure the speed with quantization.

In [31]:
%%time

with torch.no_grad():
    output = model_int8(batch['input_ids'], batch['attention_mask'])

CPU times: user 33.1 s, sys: 6.84 s, total: 40 s
Wall time: 20.4 s


We can see that it has become faster, although not significantly.

We run the code on the CPU because torch does not support the use of GPUs with quantized parameters:

In [28]:
# The demonstration needs a CUDA device; on a CPU-only machine there is nothing to move the model to.
if torch.cuda.is_available():
    try:
        model_int8.cuda()
        with torch.no_grad():
            output = model_int8(batch['input_ids'].cuda(), batch['attention_mask'].cuda())
    except (NotImplementedError, RuntimeError) as e:
        print(e)
else:
    print("No CUDA device: on a GPU the quantized linear layers raise NotImplementedError here.")

Could not run 'quantized::linear_dynamic' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::linear_dynamic' is only available for these backends: [CPU, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMeta, Tracer, AutocastCPU, AutocastCUDA, FuncTorchBatched, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFrontMode, PreDispatch, PythonDispatcher].

CPU: registered at ../aten/src/ATen/native/quantized/cpu/qlinear_dynamic.cpp:662 [kernel]
BackendSelect: fallthrough registered at

So, we do not recommend to use torch quantization, since it doesn't work on GPU, has types issues and more and more.

__Important note__: The runtime of quantized models in torch depends heavily on your hardware and OS. In some cases, quantization can significantly slow down the model. In addition, different versions of torch handle quantization differently. __In general, it is better to use other libraries.__

### BitsAndBytes

Quantization methods from huggingface are much more practical. All you need to do is specify the configuration and pass it when loading the model. The model will load automatically on the desired device with the required weight types.

In [ ]:
# needs `pip install bitsandbytes` and an NVIDIA GPU
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, quantization_config=quantization_config, num_labels=2
)

In [15]:
next(model.parameters()).device

device(type='cuda', index=0)

All non-quantized weights are stored in fp16.

In [16]:
print('Memory usage: %.3f GB' % (model.get_memory_footprint() / 2**30))

Модель занимает 0.085 GB


In [17]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear8bitLt(in_features=768, out_features=768, bias=True)
            (k_lin): Linear8bitLt(in_features=768, out_features=768, bias=True)
            (v_lin): Linear8bitLt(in_features=768, out_features=768, bias=True)
            (out_lin): Linear8bitLt(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout

In [ ]:
%%time

# the 8-bit model lives on the GPU, so the batch goes there too
with torch.no_grad():
    output = model(batch['input_ids'].to(model.device), batch['attention_mask'].to(model.device))

In [21]:
evaluate(model)

100%|██████████| 391/391 [03:50<00:00,  1.70it/s]


tensor(0.9266)

This quantization accelerated the model's application by more than 1.5 times without any loss in quality.

In addition to BitsAndBytes, there are many other quantization modules available. The complete list can be found [here](https://huggingface.co/docs/transformers/main/en/quantization/overview): you can find which method supports CPU / GPU / MPS and which doesn't, the types avaliable, dynamic / static quantization.

## Static quantization

Static quantization fixes the activation ranges in advance on a calibration set, so that the whole graph runs in integers. The recipe below uses ONNX Runtime through `optimum`; it runs on the CPU.

In [ ]:
import platform

from optimum.onnxruntime import ORTQuantizer, ORTModelForSequenceClassification
from optimum.onnxruntime.configuration import AutoQuantizationConfig, AutoCalibrationConfig
from optimum.onnxruntime.preprocessors import QuantizationPreprocessor
from optimum.onnxruntime.preprocessors.passes import (
    ExcludeGeLUNodes, ExcludeLayerNormNodes, ExcludeNodeAfter, ExcludeNodeFollowedBy,
)

# export the fine-tuned model to ONNX (CPU); static quantization will calibrate activation ranges on real inputs
onnx_model = ORTModelForSequenceClassification.from_pretrained(CHECKPOINT, export=True)
quantizer = ORTQuantizer.from_pretrained(onnx_model)
# the int8 kernels differ per CPU: arm64 on Apple silicon, avx512_vnni (or avx2) on x86
if platform.machine() in ("arm64", "aarch64"):
    qconfig = AutoQuantizationConfig.arm64(is_static=True, per_channel=False)
else:
    qconfig = AutoQuantizationConfig.avx512_vnni(is_static=True, per_channel=False)

In [31]:
qconfig

QuantizationConfig(is_static=True, format=<QuantFormat.QDQ: 1>, mode=<QuantizationMode.QLinearOps: 1>, activations_dtype=<QuantType.QUInt8: 1>, activations_symmetric=False, weights_dtype=<QuantType.QInt8: 0>, weights_symmetric=True, per_channel=False, reduce_range=False, nodes_to_quantize=[], nodes_to_exclude=[], operators_to_quantize=['Conv', 'ConvTranspose', 'Gemm', 'Clip', 'Relu', 'Reshape', 'Transpose', 'Squeeze', 'Unsqueeze', 'Resize', 'MaxPool', 'AveragePool', 'MatMul', 'Split', 'Gather', 'Softmax', 'Where', 'InstanceNormalization'], qdq_add_pair_to_weight=False, qdq_dedicated_pair=False, qdq_op_type_per_channel_support_to_axis={'MatMul': 1})

In [ ]:
# a few hundred tokenized examples are enough to estimate the [a, b] ranges of every layer
calibration_dataset = tokenized_imdb.remove_columns("label").select(range(200)).map(
    lambda batch: tokenizer.pad(batch, padding="max_length", max_length=256), batched=True
)

In [ ]:
calibration_config = AutoCalibrationConfig.minmax(calibration_dataset)

# calibrate the [a, b] ranges of all layers
ranges = quantizer.fit(
    dataset=calibration_dataset,
    calibration_config=calibration_config,
    operators_to_quantize=qconfig.operators_to_quantize,
)

# Not every node survives int8: LayerNorm, GELU, the residual adds and the attention softmax
# are the usual sources of accuracy loss, so they are left in float (the standard recipe for BERT-like models).
preprocessor = QuantizationPreprocessor()
preprocessor.register_pass(ExcludeLayerNormNodes())
preprocessor.register_pass(ExcludeGeLUNodes())
preprocessor.register_pass(ExcludeNodeAfter("Add", "Add"))
preprocessor.register_pass(ExcludeNodeAfter("Gather", "Add"))
preprocessor.register_pass(ExcludeNodeFollowedBy("Add", "Softmax"))

quantized_dir = quantizer.quantize(
    save_dir="quantized_distilbert",
    calibration_tensors_range=ranges,
    quantization_config=qconfig,
    preprocessor=preprocessor,
)
quantized_dir

In [ ]:
onnx_quantized = ORTModelForSequenceClassification.from_pretrained("quantized_distilbert", file_name="model_quantized.onnx")

In [ ]:
%%time

# the ONNX Runtime model takes the same inputs; compare with the float32 and dynamic-int8 timings above
output = onnx_quantized(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])

In [ ]:
# the ORT model is called with keyword arguments and returns logits on the CPU
correct = 0
for batch in tqdm(dataloader):
    logits = onnx_quantized(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
    correct += (batch['labels'] == logits.argmax(-1)).float().sum()
print('accuracy: %.4f' % (correct / len(dataloader.dataset)))

The accuracy cell above is the honest part of static quantization: with plain min-max calibration the int8 model loses several points on IMDB (around 0.80 against 0.91 for the float model on this subset), even with LayerNorm / GELU / softmax left in float. Static int8 is the cheapest to run (no range computation at inference), but it needs a careful calibration recipe: percentile or entropy calibration, per-channel weights, or quantization-aware training. Dynamic int8 (the same `AutoQuantizationConfig` with `is_static=False`, as in the torch example above) computes activation ranges on the fly and keeps the accuracy of the float model on this task.

In [ ]:
# dynamic int8 through the same ONNX Runtime pipeline, for comparison: no calibration, ranges computed at inference
dynamic_config = (AutoQuantizationConfig.arm64 if platform.machine() in ("arm64", "aarch64") else AutoQuantizationConfig.avx512_vnni)(is_static=False, per_channel=False)
quantizer.quantize(save_dir="quantized_distilbert_dynamic", quantization_config=dynamic_config)
onnx_dynamic = ORTModelForSequenceClassification.from_pretrained("quantized_distilbert_dynamic", file_name="model_quantized.onnx")

correct = 0
for batch in tqdm(dataloader):
    logits = onnx_dynamic(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
    correct += (batch['labels'] == logits.argmax(-1)).float().sum()
print('accuracy (dynamic int8): %.4f' % (correct / len(dataloader.dataset)))

### Fine-tuning quantized models

It is often useful to fine-tune models. Sometimes these models are so large that they can only fit into memory in quantized form. In this case, quantization must also be used during fine-tuning. All higgingface modules do not support fine-tuning quantized weights due to its instability. However, it is possible to fine-tune a quantized adapter, most often using [QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes).
Below is an example of how this can be done.

In [1]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,  # квантизуем все веса в 4 бита
    bnb_4bit_quant_type="nf4",  # (4-bit NormalFloat) специальный тип переменных,
                                # который подходит для нормально распредененных весов
    bnb_4bit_use_double_quant=True,  # дополнительно квантизуем константы квантизации
    bnb_4bit_compute_dtype=torch.bfloat16,  # операции производятся в bfloat16 для ускорения
)

/home/amshabalin/anaconda3/envs/karpov/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import AutoModelForCausalLM

# Mistral is an example; any causal LM works. The 7B checkpoint needs ~5 GB of GPU memory in 4 bits.
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1", quantization_config=quantization_config, device_map="auto")

Call `prepare_model_for_kbit_training` for the model. This is necessary to set the types of all weights, in particular:
1. Convert all unquantized weights to fp32: Layer Norm, weights of the last linear layer.
2. Add `requires_grad=True` for embeddings.

In [5]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig

config = LoraConfig(
    r=16,
    target_modules="all-linear",
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

In [ ]:
from peft import get_peft_model

model = get_peft_model(model, config)
model.print_trainable_parameters()

This model can then be trained using `Trainer` as usual.